In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
# 【关键两行】——必须同时设置！
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'PingFang SC', 'Heiti TC', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题[1][2][3][4][5][6][7][8][9][10]

import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/聚宽数据')
import datasource 
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.getAStockCodes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:4938,第一支股票:000001.XSHE


index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-06      0  6.45  6.58  6.33   6.49  8.520715e+08   
            2018-07-09      1  6.51  6.76  6.50   6.76  1.255008e+09   
            2018-07-10      2  6.76  6.76  6.66   6.73  8.030390e+08   
            2018-07-11      3  6.56  6.61  6.50   6.58  7.447658e+08   
            2018-07-12      4  6.54  6.82  6.52   6.75  1.008659e+09   

                             volume  
code        date                     
000001.XSHE 2018-07-06  131957580.0  
            2018-07-09  188260099.0  
            2018-07-10  119750900.0  
            2018-07-11  113666923.0  
            2018-07-12  149984167.0

# 从收盘价分析

In [2]:
dt['is_down'] = dt['close'].diff().lt(0)
dt['group_id_down'] = (~dt['is_down']).cumsum()
dt['streak_down'] = dt['is_down'].groupby(dt['group_id_down']).cumsum().astype(int)
# 我这里只是取得下一日是0的
dt['next_streak_down'] = dt['streak_down'].shift(-1) # 取得下一日的
streak_down = dt.loc[dt['next_streak_down'] == 0, 'streak_down'].to_list()

In [3]:
print(f'股票数据数量:{len(dt)}, 取得下跌批次:{len(streak_down)}, 最大值:{max(streak_down)}, 平均值：{sum(streak_down)/len(streak_down):.0f}')

股票数据数量:2000, 取得下跌批次:984, 最大值:10, 平均值：1


In [4]:
dt['is_up'] = dt['close'].diff().gt(0)
dt['group_id_up'] = (~dt['is_up']).cumsum()
dt['streak_up'] = dt['is_up'].groupby(dt['group_id_up']).cumsum().astype(int)
# 我这里只是取得下一日是0的
dt['next_streak_up'] = dt['streak_up'].shift(-1) # 取得下一日的
streak_up = dt.loc[dt['next_streak_up'] == 0, 'streak_up'].to_list()
print(f'股票数据数量:{len(dt)}, 取得下跌批次:{len(streak_up)}, 最大值:{max(streak_up)}, 平均值：{sum(streak_up)/len(streak_up):.0f}')

股票数据数量:2000, 取得下跌批次:1080, 最大值:10, 平均值：1


上边看起来平均值是1，可能是有大量的0吧，这里筛选0试试

In [5]:
streak_down_2 = list(filter(lambda x: x> 0, streak_down))
print(f' 取得下跌批次:{len(streak_down_2)}, 最大值:{max(streak_down_2)}, 平均值：{sum(streak_down_2)/len(streak_down_2):.0f}')
streak_up_2 = list(filter(lambda x: x> 0, streak_up))
print(f' 取得下跌批次:{len(streak_up_2)}, 最大值:{max(streak_up_2)}, 平均值：{sum(streak_up_2)/len(streak_up_2):.0f}')

 取得下跌批次:509, 最大值:10, 平均值：2
 取得下跌批次:502, 最大值:10, 平均值：2


# 从均线分析

In [6]:
def streak_ma(dt:pd.DataFrame, days:int, is_print:bool=True):
    """均线的连续性"""
    # 均线
    dt[f'Ma{days}'] = talib.SMA(dt['close'], timeperiod=days) 
    # 下降
    dt[f'is_down_{days}'] = dt[f'Ma{days}'].diff().lt(0)
    dt[f'group_id_down_{days}'] = (~dt[f'is_down_{days}']).cumsum()
    dt[f'streak_down_{days}'] = dt[f'is_down_{days}'].groupby(dt[f'group_id_down_{days}']).cumsum().astype(int)
    # 我这里只是取得下一日是0的
    dt[f'next_streak_down_{days}'] = dt[f'streak_down_{days}'].shift(-1) # 取得下一日的
    streak_down = dt.loc[dt[f'next_streak_down_{days}'] == 0, f'streak_down_{days}'].to_list()
    # 张张
    dt[f'is_up_{days}'] = dt[f'Ma{days}'].diff().gt(0)
    dt[f'group_id_up_{days}'] = (~dt[f'is_up_{days}']).cumsum()
    dt[f'streak_up_{days}'] = dt[f'is_up_{days}'].groupby(dt[f'group_id_up_{days}']).cumsum().astype(int)
    # 我这里只是取得下一日是0的
    dt[f'next_streak_up_{days}'] = dt[f'streak_up_{days}'].shift(-1) # 取得下一日的
    streak_up = dt.loc[dt[f'next_streak_up_{days}'] == 0, f'streak_up_{days}'].to_list()
    if is_print:
        print(f'股票数据数量:{len(dt):5d}，取得上涨批次:{len(streak_up):5d}, 最大值:{max(streak_up):5d}, 平均值：{sum(streak_up)/len(streak_up):.1f}, 取得下跌批次:{len(streak_down):5d}, 最大值:{max(streak_down):5d}, 平均值：{sum(streak_down)/len(streak_down):.1f}, ')
    return (streak_up, streak_down)
    

In [7]:
streak_ma(dt, 3)
streak_ma(dt, 5)
streak_ma(dt, 10)
streak_ma(dt, 20)
pass

股票数据数量: 2000，取得上涨批次: 1050, 最大值:   15, 平均值：0.9, 取得下跌批次:  991, 最大值:   20, 平均值：1.0, 
股票数据数量: 2000，取得上涨批次: 1043, 最大值:   23, 平均值：0.9, 取得下跌批次:  982, 最大值:   21, 平均值：1.0, 
股票数据数量: 2000，取得上涨批次: 1021, 最大值:   51, 平均值：1.0, 取得下跌批次: 1006, 最大值:   34, 平均值：1.0, 
股票数据数量: 2000，取得上涨批次: 1019, 最大值:   54, 平均值：1.0, 取得下跌批次: 1013, 最大值:   67, 平均值：1.0, 


大概率不用全部股票分析了

In [8]:
streak_3 = streak_ma(dt, 3)
streak_3_up_2 = [i for i in streak_3[0] if i > 0]
streak_3_down_2 = [i for i in streak_3[1] if i > 0]
print(f'上涨批次:{len(streak_3_up_2)}, 最大值:{max(streak_3_up_2)}, 平均值：{sum(streak_3_up_2)/len(streak_3_up_2):.0f}, 下降批次:{len(streak_3_down_2)}, 最大值:{max(streak_3_down_2)}, 平均值：{sum(streak_3_down_2)/len(streak_3_down_2):.0f}, ')

股票数据数量: 2000，取得上涨批次: 1050, 最大值:   15, 平均值：0.9, 取得下跌批次:  991, 最大值:   20, 平均值：1.0, 
上涨批次:263, 最大值:15, 平均值：4, 下降批次:270, 最大值:20, 平均值：4, 


In [9]:
for day in [3,5,10,20,30,60]:
    print(f'均线:{day}' + '#'*10)
    _streak = streak_ma(dt, day)
    _streak_up_2 = [i for i in _streak[0] if i > 0]
    _streak_down_2 = [i for i in _streak[1] if i > 0]
    print(f'上涨批次:{len(_streak_up_2)}, 最大值:{max(_streak_up_2)}, 平均值：{sum(_streak_up_2)/len(_streak_up_2):.0f}, 下降批次:{len(_streak_down_2)}, 最大值:{max(_streak_down_2)}, 平均值：{sum(_streak_down_2)/len(_streak_down_2):.0f}, ')
    


均线:3##########
股票数据数量: 2000，取得上涨批次: 1050, 最大值:   15, 平均值：0.9, 取得下跌批次:  991, 最大值:   20, 平均值：1.0, 
上涨批次:263, 最大值:15, 平均值：4, 下降批次:270, 最大值:20, 平均值：4, 
均线:5##########
股票数据数量: 2000，取得上涨批次: 1043, 最大值:   23, 平均值：0.9, 取得下跌批次:  982, 最大值:   21, 平均值：1.0, 
上涨批次:199, 最大值:23, 平均值：5, 下降批次:196, 最大值:21, 平均值：5, 
均线:10##########
股票数据数量: 2000，取得上涨批次: 1021, 最大值:   51, 平均值：1.0, 取得下跌批次: 1006, 最大值:   34, 平均值：1.0, 
上涨批次:148, 最大值:51, 平均值：7, 下降批次:144, 最大值:34, 平均值：7, 
均线:20##########
股票数据数量: 2000，取得上涨批次: 1019, 最大值:   54, 平均值：1.0, 取得下跌批次: 1013, 最大值:   67, 平均值：1.0, 
上涨批次:83, 最大值:54, 平均值：12, 下降批次:81, 最大值:67, 平均值：12, 
均线:30##########
股票数据数量: 2000，取得上涨批次:  985, 最大值:   70, 平均值：1.0, 取得下跌批次: 1051, 最大值:   95, 平均值：0.9, 
上涨批次:75, 最大值:70, 平均值：13, 下降批次:75, 最大值:95, 平均值：13, 
均线:60##########
股票数据数量: 2000，取得上涨批次: 1046, 最大值:  125, 平均值：0.9, 取得下跌批次: 1028, 最大值:  118, 平均值：0.9, 
上涨批次:52, 最大值:125, 平均值：17, 下降批次:55, 最大值:118, 平均值：18, 


可以看到去掉0后，明显平均值扩大了

In [10]:
for day in [3,5,10,20,30,60]:
    print(f'均线:{day}' + '#'*10)
    _streak = streak_ma(dt, day)
    _streak_up_2 = [i for i in _streak[0] if i > 4]
    _streak_down_2 = [i for i in _streak[1] if i > 4]
    print(f'上涨批次:{len(_streak_up_2)}, 最大值:{max(_streak_up_2)}, 平均值：{sum(_streak_up_2)/len(_streak_up_2):.0f}, 下降批次:{len(_streak_down_2)}, 最大值:{max(_streak_down_2)}, 平均值：{sum(_streak_down_2)/len(_streak_down_2):.0f}, ')

均线:3##########
股票数据数量: 2000，取得上涨批次: 1050, 最大值:   15, 平均值：0.9, 取得下跌批次:  991, 最大值:   20, 平均值：1.0, 
上涨批次:74, 最大值:15, 平均值：7, 下降批次:75, 最大值:20, 平均值：8, 
均线:5##########
股票数据数量: 2000，取得上涨批次: 1043, 最大值:   23, 平均值：0.9, 取得下跌批次:  982, 最大值:   21, 平均值：1.0, 
上涨批次:91, 最大值:23, 平均值：8, 下降批次:91, 最大值:21, 平均值：9, 
均线:10##########
股票数据数量: 2000，取得上涨批次: 1021, 最大值:   51, 平均值：1.0, 取得下跌批次: 1006, 最大值:   34, 平均值：1.0, 
上涨批次:66, 最大值:51, 平均值：13, 下降批次:63, 最大值:34, 平均值：13, 
均线:20##########
股票数据数量: 2000，取得上涨批次: 1019, 最大值:   54, 平均值：1.0, 取得下跌批次: 1013, 最大值:   67, 平均值：1.0, 
上涨批次:47, 最大值:54, 平均值：20, 下降批次:45, 最大值:67, 平均值：20, 
均线:30##########
股票数据数量: 2000，取得上涨批次:  985, 最大值:   70, 平均值：1.0, 取得下跌批次: 1051, 最大值:   95, 平均值：0.9, 
上涨批次:35, 最大值:70, 平均值：26, 下降批次:37, 最大值:95, 平均值：24, 
均线:60##########
股票数据数量: 2000，取得上涨批次: 1046, 最大值:  125, 平均值：0.9, 取得下跌批次: 1028, 最大值:  118, 平均值：0.9, 
上涨批次:21, 最大值:125, 平均值：40, 下降批次:25, 最大值:118, 平均值：37, 


前面除去0的平均值是4，这里注意一下，0表示是反方向的，这里再除去4，平均值竟然大幅扩大了，是否可以认为只要超过4，就会有大概率超过7？  
可以认为均线有很大的惯性，这里下单的时候可以用一下。